# TP1 â€” Building a ReAct Agent from Scratch
## Module 2IAiasd 9.7 â€” Agentic AI â€” IMT Mines AlÃ¨s

**Prerequisites:** You have completed TD1 (Exercises 0â€“4).  
Your Ollama server is still running and `qwen2.5:3b` is available.

**Objective:** Implement the full **ReAct loop** (Reason + Act) introduced in CM1,  
using the tools and concepts you practised in TD1.  
By the end of this session you will have a working agent that solves multi-step problems autonomously.



---

### Structure

| # | Exercise | Concepts |
|---|---------|----------|
| 1 | The ReAct loop â€” step by step | orchestrator, tool dispatch, ToolMessage |
| 2 | A complete math agent | multi-step reasoning, safety limit |
| 3 | Mixed-tool agent | combining tool types, robustness |
| 4 | Your own agent (open-ended) | design + evaluation |

> ðŸ“– **Reference:** CM1 slides â€” perception â†’ reasoning â†’ action â†’ observation loop  
> ðŸ“– **Reference:** TD1 Exercise 4 â€” `@tool`, `bind_tools`, `ToolMessage`  


---
## Setup â€” Carry over from TD1

Run this cell to restore the TD1 environment.  
Make sure `ollama serve` is still running in your terminal.

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
import requests, os

MODEL = "qwen2.5:3b"

# Quick health check
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"âœ… Ollama running | available models: {models}")
except Exception:
    print("âŒ Ollama not reachable â€” run: ollama serve")

llm = ChatOllama(model=MODEL, temperature=1.0)
print(f"âœ… LLM ready: {MODEL}")

# Optional: LangSmith tracing (from TD1)
# os.environ["LANGSMITH_API_KEY"]    = "lsv2_pt_..."
# os.environ["LANGSMITH_TRACING_V2"] = "true"
# os.environ["LANGSMITH_PROJECT"]    = "2IAiasd-9.7-TP1"

âœ… Ollama running | available models: ['qwen2.5:3b']
âœ… LLM ready: qwen2.5:3b


---
## Exercise 1 â€” The ReAct Loop, Step by Step

In CM1 you saw the agent loop conceptually:
```
Perceive â†’ Reason (LLM) â†’ Act (tool call) â†’ Observe (tool result) â†’ loop
```

In TD1 Exercise 4 you made **one** tool call manually.  
Now you build the **full loop** that keeps running until the LLM decides it is done.

### 1.1 â€” Trace what the loop sees at each step

In [3]:
# Tools from TD1 Exercise 4 â€” reload them here
@tool
def get_word_count(text: str) -> int:
    """Count the number of words in a given text string."""
    return len(text.split())

@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature value from Celsius to Fahrenheit."""
    return round(celsius * 9 / 5 + 32, 2)

@tool
def get_current_date() -> str:
    """Return today's date in ISO format YYYY-MM-DD."""
    from datetime import date
    return date.today().isoformat()

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index given weight in kilograms and height in metres."""
    return round(weight_kg / (height_m ** 2), 2)

tools_ex1 = [get_word_count, celsius_to_fahrenheit, get_current_date, calculate_bmi]
tools_map_ex1 = {t.name: t for t in tools_ex1}

llm_with_tools = llm.bind_tools(tools_ex1)
print(f"Available tools: {[t.name for t in tools_ex1]}")

Available tools: ['get_word_count', 'celsius_to_fahrenheit', 'get_current_date', 'calculate_bmi']


In [5]:
# Trace exactly what happens at each step for the query:
# "What is today's date and how many words are in 'Hello from IMT Mines Ales in France'?"

messages = [
    SystemMessage(content="You are a helpful assistant. Use tools when needed."),
    HumanMessage(content="What is today's date and how many words are in 'Hello from IMT Mines Ales in France'?"),
]

print("=" * 60)
print(f"INITIAL STATE: {len(messages)} messages")
for m in messages:
    print(f"  {m.type}: {m.content[:60]}")
print("=" * 60)

step = 0
while True:
    step += 1
    print(f"\n--- STEP {step}: REASON ---")

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    print(f"  LLM content : {repr(response.content[:80]) if response.content else '(empty)'}")
    print(f"  Tool calls  : {response.tool_calls}")
    print(f"  Msg count   : {len(messages)}")

    if not response.tool_calls:
        print("\n--- LOOP EXITS: no more tool calls ---")
        break

    print(f"\n--- STEP {step}: ACT + OBSERVE ---")
    for tc in response.tool_calls:
        fn = tools_map_ex1[tc["name"]]
        result = fn.invoke(tc["args"])
        tool_msg = ToolMessage(content=str(result), tool_call_id=tc["id"])
        messages.append(tool_msg)
        print(f"  {tc['name']}({tc['args']}) â†’ {result}")

print(f"\n{'=' * 60}")
print(f"FINAL ANSWER: {response.content}")
print(f"Total steps: {step} | Total messages: {len(messages)}")

INITIAL STATE: 2 messages
  system: You are a helpful assistant. Use tools when needed.
  human: What is today's date and how many words are in 'Hello from I

--- STEP 1: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : (empty)
  Tool calls  : [{'name': 'get_current_date', 'args': {}, 'id': 'cb2368fd-6517-45fd-a57d-52d0cc24308d', 'type': 'tool_call'}, {'name': 'get_word_count', 'args': {'text': 'Hello from IMT Mines Ales in France'}, 'id': '00a2a38f-5964-4b32-ab54-3eac77f675d4', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  get_current_date({}) â†’ 2026-10-16
  get_word_count({'text': 'Hello from IMT Mines Ales in France'}) â†’ 7

--- STEP 2: REASON ---
  LLM content : "Today's date is 2026-10-16 and the sentence 'Hello from IMT Mines Ales in France"
  Tool calls  : []
  Msg count   : 6

--- LOOP EXITS: no more tool calls ---

FINAL ANSWER: Today's date is 2026-10-16 and the sentence 'Hello from IMT Mines Ales in France' contains 7 words.
Total steps: 2 | Total messages: 6


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


### 1.2 â€” âœï¸ Your turn: trace analysis

Answer the following questions **in the Markdown cell below**,  
based on the trace you just observed. Be precise and reference specific message types.

**Answers:**

1. How many tool calls did the model make? Which tools were called and in what order?
   > The model made **2 tool calls** in total, issued in **parallel** within a single turn during Step 1:
   > 1. `get_current_date` with arguments `{}`
   > 2. `get_word_count` with arguments `{'text': 'Hello from IMT Mines Ales in France'}`

2. What does `response.content` contain when the model decides to call a tool vs. when it gives the final answer?
   > - **When calling a tool (Step 1):** `response.content` is **empty** (`""` / `(empty)`). The model's intent is stored entirely inside the `response.tool_calls` list of the returned `AIMessage`.
   > - **When giving the final answer (Step 2):** `response.content` contains the complete **natural language text response** summarizing the findings, while `response.tool_calls` becomes an empty list (`[]`).

3. What is the stopping condition of the loop? Why is it reliable?
   > - **Stopping Condition:** `if not response.tool_calls:` (exits when `response.tool_calls` is empty).
   > - **Why it is reliable:** The LLM explicitly signals whether it requires additional external context or has enough information to fulfill the request. Once the accumulated `ToolMessage`s provide the necessary data, the model stops requesting tools and generates the final text in `AIMessage.content`. Caveat: a confused model could get stuck by repeatedly calling the wrong tools (tools that fail to produce the correct answer to the query). Consequently, the absence of a safety parameter (such as a maximum number of iterations) could lead to an infinite loop.

4. What would happen if you removed the `SystemMessage`? Try it and report.
   > - **Observed Behavior:** The agent still executed successfully, calling both tools (`get_current_date` and `get_word_count`) in parallel and returning the exact same correct answer in 2 steps. The total message count decreased from 6 to 5 because the initial state contained only 1 message (`HumanMessage`).
   > - **Technical Reason:** `bind_tools()` injects the tool JSON schemas directly into the API invocation, so the model recognizes available tools even without an explicit `SystemMessage`.
   > - **Role of SystemMessage:** While omitting it had no impact on this simple query, a `SystemMessage` remains essential in complex agent workflows to define persona, impose operational constraints, and enforce output formatting.

---
## Exercise 2 â€” A Complete Math Agent

Now you build a **reusable `react_agent` function** that cleanly encapsulates the loop.  
This is exactly the pattern used by every production agent framework (LangGraph, CrewAI, AutoGenâ€¦).

### 2.1 â€” Define the math tools

In [2]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b. Raises ValueError if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return round(a / b, 6)

@tool
def power(base: float, exponent: float) -> float:
    """Raise base to the power of exponent."""
    return base ** exponent

@tool
def square_root(x: float) -> float:
    """Return the square root of x. Raises ValueError if x is negative."""
    import math
    if x < 0:
        raise ValueError("Cannot compute the square root of a negative number.")
    return round(math.sqrt(x), 6)

math_tools = [add, subtract, multiply, divide, power, square_root]
print(f"Math tools: {[t.name for t in math_tools]}")

Math tools: ['add', 'subtract', 'multiply', 'divide', 'power', 'square_root']


### 2.2 â€” âœï¸ Implement `react_agent`

Complete the function below. Requirements:
- Accepts `query` (str), `tools` (list), `system_prompt` (str, optional), `max_steps` (int, default 10), `verbose` (bool, default True)
- Returns the final answer as a string
- In verbose mode: prints each step's tool calls and observations
- Handles tool execution errors gracefully (catch exceptions, pass the error message as a `ToolMessage`)
- Stops if `max_steps` is reached and returns a timeout message

In [4]:
def react_agent(
    query: str,
    tools: list,
    system_prompt: str = "You are a helpful assistant. Use tools when needed to answer accurately.",
    max_steps: int = 10,
    verbose: bool = True,
) -> str:
    """
    A minimal ReAct (Reason + Act) agent.

    Args:
        query:         The user's question or task.
        tools:         List of @tool functions available to the agent.
        system_prompt: Instructions given to the LLM at the start of the conversation.
        max_steps:     Safety limit on the number of reasoning steps.
        verbose:       If True, print each step's details.

    Returns:
        The final answer as a string.
    """
    # TODO: implement this function
    # Hints:
    #   1. Build the initial messages list (SystemMessage + HumanMessage)
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=query),
    ]

    #   2. Bind the tools to the LLM with bind_tools
    llm_with_tools = llm.bind_tools(tools)

    #   3. Build a tools_map dict: {tool_name: tool_function}
    tools_map = {t.name: t for t in tools}

    #   4. Loop: invoke LLM â†’ check tool_calls â†’ execute â†’ append â†’ repeat
    steps = 0
    while steps < max_steps:
        steps += 1
        if verbose:
            print(f"\n--- STEP {steps}: REASON ---")

        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if verbose:
            print(f"  LLM content : {repr(response.content[:80]) if response.content else '(empty)'}")
            print(f"  Tool calls  : {response.tool_calls}")
            print(f"  Msg count   : {len(messages)}")

        if not response.tool_calls:
            if verbose:
                print("\n--- LOOP EXITS: no more tool calls ---")
            return response.content

        if verbose:
            print(f"\n--- STEP {steps}: ACT + OBSERVE ---")
        for tc in response.tool_calls:
            #   5. Handle tool execution errors with try/except
            try:
                fn = tools_map[tc["name"]]
                result = fn.invoke(tc["args"])
                tool_msg = ToolMessage(content=str(result), tool_call_id=tc["id"])
                messages.append(tool_msg)
                if verbose:
                    print(f"  {tc['name']}({tc['args']}) â†’ {result}")
            except Exception as e:
                error_msg = ToolMessage(content=f"Error: {str(e)}", tool_call_id=tc["id"])
                messages.append(error_msg)
                if verbose:
                    print(f"  {tc['name']}({tc['args']}) â†’ Error: {str(e)}")

    #   6. Return the final answer, or a timeout message if max_steps is reached
    return "Timeout: maximum steps reached."

In [21]:
# Test 1: simple two-step calculation
system_prompt = "You are a math expert. Answer the questions precisely and use the provided tools for calculations when needed."
result = react_agent(query="What is (3 + 5) * 12?", system_prompt=system_prompt, tools=math_tools)
print(f"\nResult: {result}")
assert "96" in result, "Expected 96 in the answer"
print("âœ… Test 1 passed")


--- STEP 1: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : (empty)
  Tool calls  : [{'name': 'add', 'args': {'b': 5, 'a': 3}, 'id': '60b98c26-28a2-4d72-b1c5-c5830cc857c1', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 8, 'b': 12}, 'id': 'e76571f8-b431-4472-aecf-39be8aa42f1e', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  add({'b': 5, 'a': 3}) â†’ 8.0
  multiply({'a': 8, 'b': 12}) â†’ 96.0

--- STEP 2: REASON ---
  LLM content : 'The result of (3 + 5) * 12 is 96.0.'
  Tool calls  : []
  Msg count   : 6

--- LOOP EXITS: no more tool calls ---

Result: The result of (3 + 5) * 12 is 96.0.
âœ… Test 1 passed


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [22]:
# Test 2: multi-step with power and square root
result = react_agent(
    query="Calculate: 2 to the power of 8, then subtract 56, then take the square root.",
    system_prompt=system_prompt,
    tools=math_tools
)
print(f"\nResult: {result}")
# 2^8 = 256, 256 - 56 = 200, sqrt(200) â‰ˆ 14.142
assert "14" in result, "Expected ~14.14 in the answer"
print("âœ… Test 2 passed")


--- STEP 1: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : (empty)
  Tool calls  : [{'name': 'power', 'args': {'base': 2, 'exponent': 8}, 'id': '18cfb8ba-e67f-4689-924a-1817fc22c499', 'type': 'tool_call'}, {'name': 'power', 'args': {'a': 256, 'b': 56}, 'id': '93fe30cb-3e29-464d-a79a-254100afc1b0', 'type': 'tool_call'}, {'name': 'square_root', 'args': {'x': 200}, 'id': '2a41316e-10a2-41da-b81f-9f1724da60a8', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  power({'base': 2, 'exponent': 8}) â†’ 256.0
  power({'a': 256, 'b': 56}) â†’ Error: 2 validation errors for power
base
  Field required [type=missing, input_value={'a': 256, 'b': 56}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
exponent
  Field required [type=missing, input_value={'a': 256, 'b': 56}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
  square_root({'x': 200}) â†’ 14.142136

--- STEP 2: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : (empty)
  Tool calls  : [{'name': 'power', 'args': {'base': 2, 'exponent': 8}, 'id': 'b46cfcb5-f660-4ce3-a3b4-d3b06b29a721', 'type': 'tool_call'}, {'name': 'subtract', 'args': {'a': 256, 'b': 56}, 'id': '0dfc9b2a-f11c-46f4-b5e2-54d42ce5178e', 'type': 'tool_call'}, {'name': 'square_root', 'args': {'x': 200}, 'id': '7a37df76-cdf5-449b-bd69-d1dc0bbf8a6b', 'type': 'tool_call'}]
  Msg count   : 7

--- STEP 2: ACT + OBSERVE ---
  power({'base': 2, 'exponent': 8}) â†’ 256.0
  subtract({'a': 256, 'b': 56}) â†’ 200.0
  square_root({'x': 200}) â†’ 14.142136

--- STEP 3: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : "It seems there was a misunderstanding in the sequence of operations. Let's corre"
  Tool calls  : [{'name': 'square_root', 'args': {'x': 200}, 'id': 'b1bab89c-48aa-406f-a475-da4e4da7a1fd', 'type': 'tool_call'}]
  Msg count   : 11

--- STEP 3: ACT + OBSERVE ---
  square_root({'x': 200}) â†’ 14.142136

--- STEP 4: REASON ---
  LLM content : 'The square root of 200.0 is 14.142136.\n\nTherefore, the final result is 14.142136'
  Tool calls  : []
  Msg count   : 13

--- LOOP EXITS: no more tool calls ---

Result: The square root of 200.0 is 14.142136.

Therefore, the final result is 14.142136.
âœ… Test 2 passed


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [23]:
# Test 3: error handling â€” division by zero
result = react_agent(query="What is 42 divided by 0?", system_prompt=system_prompt, tools=math_tools)
print(f"\nResult: {result}")
# The agent should receive the error and report it gracefully without crashing
print("âœ… Test 3 â€” verify the agent handled the error without raising an exception")


--- STEP 1: REASON ---
  LLM content : (empty)
  Tool calls  : [{'name': 'divide', 'args': {'a': 42, 'b': 0}, 'id': '2946b7af-bfab-4758-a024-38fba3645e4a', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  divide({'a': 42, 'b': 0}) â†’ Error: Cannot divide by zero.

--- STEP 2: REASON ---
  LLM content : 'The operation of dividing 42 by 0 is not valid as division by zero is undefined '
  Tool calls  : []
  Msg count   : 5

--- LOOP EXITS: no more tool calls ---

Result: The operation of dividing 42 by 0 is not valid as division by zero is undefined in mathematics.
âœ… Test 3 â€” verify the agent handled the error without raising an exception


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [25]:
# Test 4: no tool needed
result = react_agent(query="What is the Pythagorean theorem?", system_prompt=system_prompt, tools=math_tools, verbose=False)
print(f"Result: {result}")
print("âœ… Test 4 â€” agent answered without calling any tool")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Result: The Pythagorean theorem is a fundamental principle in Euclidean geometry that relates to right-angled triangles. It states that in a right-angled triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides (known as the legs). Mathematically, this is expressed as \( c^2 = a^2 + b^2 \), where \( c \) is the length of the hypotenuse, and \( a \) and \( b \) are the lengths of the other two sides.

I can help you perform calculations related to this theorem, if you need to find the length of one side given the lengths of the other two sides, or to find the length of the hypotenuse given the lengths of the legs. Would you like to proceed with a calculation?
âœ… Test 4 â€” agent answered without calling any tool


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


---
## Exercise 3 â€” Mixed-Tool Agent

A real-world agent needs tools of **different types** â€” not just one category.  
Here you combine the math tools from Exercise 2 with the utility tools from TD1,  
and add two new tools of your own design.

### 3.1 â€” Combine the existing tools

In [5]:
# Bring back the utility tools from TD1
combined_tools = math_tools + [get_word_count, celsius_to_fahrenheit, get_current_date, calculate_bmi]
print(f"Combined tools ({len(combined_tools)}): {[t.name for t in combined_tools]}")

Combined tools (10): ['add', 'subtract', 'multiply', 'divide', 'power', 'square_root', 'get_word_count', 'celsius_to_fahrenheit', 'get_current_date', 'calculate_bmi']


### 3.2 â€” âœï¸ Add 2 new tools of your choice

Design and implement 2 new tools. Pick from the suggestions below or invent your own.  
Each tool must have:
- A clear, precise **docstring** (remember: the docstring IS the prompt sent to the LLM)
- **Type annotations** on all parameters
- At least one **edge case** handled (empty input, division by zero, negative valueâ€¦)

**Suggestions:** `reverse_string`, `is_palindrome`, `count_vowels`, `days_until_date`, `percentage(value, total)`, `fibonacci(n)`, `is_prime(n)`

In [7]:
@tool
def string_reverse(text: str) -> str:
    """Reverse the input string.

    Args:
        text (str): The string to reverse. Cannot be empty.

    Returns:
        str: The reversed string.
    """
    if not text or not text.strip():
        raise ValueError("Cannot reverse an empty or whitespace-only string.")

    return text[::-1]


@tool
def celsius_to_kelvin(celsius: float) -> float:
    """Convert a temperature value from Celsius to Kelvin.

    Args:
        celsius (float): The temperature in degrees Celsius. Must be >= -273.15.

    Returns:
        float: The temperature in Kelvin.
    """
    if celsius < -273.15:
        raise ValueError("Temperature cannot be lower than absolute zero (-273.15°C).")

    return round(celsius + 273.15, 2)

# Add your updated tools to the combined list
all_tools = combined_tools + [string_reverse, celsius_to_kelvin]
print(f"All tools ({len(all_tools)}): {[t.name for t in all_tools]}")

All tools (12): ['add', 'subtract', 'multiply', 'divide', 'power', 'square_root', 'get_word_count', 'celsius_to_fahrenheit', 'get_current_date', 'calculate_bmi', 'string_reverse', 'celsius_to_kelvin']


### 3.3 â€” âœï¸ Test with cross-category queries

Write **3 queries** that each require tools from **at least 2 different categories**  
(e.g. math + date, or string + math, or your custom tool + an existing one).

In [8]:
# Query 1: requires math + date (or math + another category)
# Example: "What is today's date, and what is 2 to the power of the day number?"
# TODO: write your own query
query1 = "What is today's date, and what is 2 to the power of 10?"
result1 = react_agent(query1, all_tools)
print(f"Q1: {query1}")
print(f"A1: {result1}\n")


--- STEP 1: REASON ---


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : (empty)
  Tool calls  : [{'name': 'get_current_date', 'args': {}, 'id': 'f0a0bfd2-279b-49f3-a157-936e95d45825', 'type': 'tool_call'}, {'name': 'power', 'args': {'base': 2, 'exponent': 10}, 'id': '183af56c-77c8-4559-ac41-1a5849bbd701', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  get_current_date({}) â†’ 2026-10-19
  power({'base': 2, 'exponent': 10}) â†’ 1024.0

--- STEP 2: REASON ---
  LLM content : "Today's date is 2026-10-19 and 2 to the power of 10 is 1024.0."
  Tool calls  : []
  Msg count   : 6

--- LOOP EXITS: no more tool calls ---
Q1: What is today's date, and what is 2 to the power of 10?
A1: Today's date is 2026-10-19 and 2 to the power of 10 is 1024.0.



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [9]:
# Query 2: requires one of your custom tools + an existing tool
# TODO: write your own query
query2 = "Convert 30 degrees Celsius to Kelvin, and after subtract 50 from the result."
result2 = react_agent(query2, all_tools)
print(f"Q2: {query2}")
print(f"A2: {result2}\n")


--- STEP 1: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : (empty)
  Tool calls  : [{'name': 'celsius_to_kelvin', 'args': {'celsius': 30}, 'id': '0d286abd-43bb-46b1-b6ba-32b61aa6c6d8', 'type': 'tool_call'}, {'name': 'subtract', 'args': {'a': 50, 'b': 'result_of_celsius_to_kelvin_function'}, 'id': 'f23ba5df-a045-49ca-8b1f-7dd3a8d6e5d5', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  celsius_to_kelvin({'celsius': 30}) â†’ 303.15
  subtract({'a': 50, 'b': 'result_of_celsius_to_kelvin_function'}) â†’ Error: 1 validation error for subtract
b
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='result_of_celsius_to_kelvin_function', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/float_parsing

--- STEP 2: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : "It seems there was an issue with the direct substitution of the result. Let's go"
  Tool calls  : [{'name': 'celsius_to_kelvin', 'args': {'celsius': 30}, 'id': 'c163d8bf-5cd6-4b80-998f-f1070f4120dd', 'type': 'tool_call'}]
  Msg count   : 6

--- STEP 2: ACT + OBSERVE ---
  celsius_to_kelvin({'celsius': 30}) â†’ 303.15

--- STEP 3: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : 'The temperature of 30 degrees Celsius is 303.15 Kelvin when converted to Kelvin.'
  Tool calls  : [{'name': 'subtract', 'args': {'a': 50, 'b': '303.15'}, 'id': '1874e129-e032-4049-a310-46f4b355edc7', 'type': 'tool_call'}]
  Msg count   : 8

--- STEP 3: ACT + OBSERVE ---
  subtract({'a': 50, 'b': '303.15'}) â†’ -253.14999999999998

--- STEP 4: REASON ---
  LLM content : 'The result of subtracting 50 from 303.15 Kelvin is approximately -253.15 Kelvin.'
  Tool calls  : []
  Msg count   : 10

--- LOOP EXITS: no more tool calls ---
Q2: Convert 30 degrees Celsius to Kelvin, and after subtract 50 from the result.
A2: The result of subtracting 50 from 303.15 Kelvin is approximately -253.15 Kelvin. 

If you have any further questions or need additional calculations, feel free to ask!



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [10]:
# Query 3: a more complex chain (3+ tool calls expected)
# TODO: write your own query
query3 = "Reverse the string 'Agentic AI'. After, find how many words are in 'LLMs are transforming software engineering' then multiply that word count by 5."
result3 = react_agent(query3, all_tools)
print(f"Q3: {query3}")
print(f"A3: {result3}\n")


--- STEP 1: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : (empty)
  Tool calls  : [{'name': 'string_reverse', 'args': {'text': 'Agentic AI'}, 'id': 'f551908c-7189-4379-be75-b7cff160c9fb', 'type': 'tool_call'}, {'name': 'get_word_count', 'args': {'text': 'LLMs are transforming software engineering'}, 'id': '6dc17e0a-8780-4a0f-9483-20a17a2b1900', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'b': 1, 'a': 5}, 'id': '8b484d7b-c864-4268-b435-a1de5fe4e76d', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  string_reverse({'text': 'Agentic AI'}) â†’ IA citnegA
  get_word_count({'text': 'LLMs are transforming software engineering'}) â†’ 5
  multiply({'b': 1, 'a': 5}) â†’ 5.0

--- STEP 2: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : "It seems there was a misunderstanding in the calculation step. Let's clarify and"
  Tool calls  : [{'name': 'multiply', 'args': {'a': 5, 'b': 5}, 'id': 'f9ae32f5-14e4-446d-9d3a-a6bd9ab63fc2', 'type': 'tool_call'}]
  Msg count   : 7

--- STEP 2: ACT + OBSERVE ---
  multiply({'a': 5, 'b': 5}) â†’ 25.0

--- STEP 3: REASON ---
  LLM content : "The calculation is now correct. The word count for the string 'LLMs are transfor"
  Tool calls  : []
  Msg count   : 9

--- LOOP EXITS: no more tool calls ---
Q3: Reverse the string 'Agentic AI'. After, find how many words are in 'LLMs are transforming software engineering' then multiply that word count by 5.
A3: The calculation is now correct. The word count for the string 'LLMs are transforming software engineering' is 5, and multiplying this by 5 gives us 25. Therefore, the final answer is:

25.0



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


### 3.4 â€” âœï¸ Reflection

**Reflection:**

1. Did the agent always call tools in the order you expected? If not, why?
   > **No, the agent frequently attempted speculative parallel tool execution.**
   > - While independent tasks in Query 1 executed smoothly in parallel (`get_current_date` + `power`), Queries 2 and 3 showed the model's eagerness to execute all steps in Step 1.
   > - In Query 2, the LLM attempted to call `subtract` simultaneously with `celsius_to_kelvin` by passing a literal string placeholder (`'result_of_celsius_to_kelvin_function'`) as argument `b`.
   > - In Query 3, it passed a dummy value (`b=1`) to `multiply` before knowing the actual word count.
   > - **Reason:** Small local models like `qwen2.5:3b` lack strict step-by-step reasoning enforcement by default and tend to resolve all detected intent slots in a single turn.

2. Did the model use your custom tools correctly? What role did the docstring play?
   > **Yes, both `celsius_to_kelvin` and `string_reverse` were invoked with correct parameter structures.**
   > - **Role of docstrings:** The docstring serves as the tool's schema definition for the LLM. It provided the exact function semantics (`"Convert a temperature value from Celsius to Kelvin"`) and argument types required by `bind_tools()`. 
   > - Because the docstrings clearly stated input expectations, the model accurately mapped user inputs (e.g., `'Agentic AI'` to `text` and `30` to `celsius`).

3. Did any query cause a wrong tool call or a wrong answer? If so, how would you fix it?
   > **Yes, Query 2 encountered both a runtime tool error and a final mathematical logic error, while Query 3 successfully self-corrected:**
   > 
   > - **Query 2 Issues:**
   >   1. *Tool Error Recovery:* In Step 1, passing the string `'result_of_celsius_to_kelvin_function'` to `subtract` triggered a Pydantic `ValidationError` (`unable to parse string as a number`). Our `react_agent` exception handler caught this and sent it as a `ToolMessage`, allowing the agent to recover and retry in Step 2.
   >   2. *Operand Inversion:* When executing `subtract`, the model passed `a=50` and `b=303.15`, resulting in $50 - 303.15 = -253.15$. To "subtract 50 from 303.15", it should have passed `a=303.15` and `b=50` ($253.15$).
   > 
   > - **Query 3 Self-Correction:**
   >   - The model guessed `multiply(a=5, b=1)` in Step 1. Upon observing `get_word_count = 5`, it recognized the discrepancy and re-issued `multiply(a=5, b=5)` in Step 2, arriving at the correct answer ($25.0$).
   > 
   > - **How to Fix:**
   >   1. **System Prompt Rules:** Explicitly instruct the model: *"Do not guess or use string placeholders for future inputs. If a tool argument depends on another tool's output, wait for the observation before invoking the second tool."*
   >   2. **Disallow Parallel Tooling:** Configure `bind_tools(parallel_tool_calls=False)` if supported by the provider, forcing single sequential steps.
   >   3. **Clarify Tool Docstrings:** Update math tool docstrings to explicitly define parameter roles (e.g., `subtract(a, b)`: computes `a - b`).

---
## Exercise 4 â€” Your Own Agent

Design and build a **domain-specific agent** of your choice.

**What to include:**
- At least **4 tools**, at least **2 original** (not reused from previous exercises)
- A carefully designed **system prompt**: role, constraints, output format
- At least **5 test queries**: normal cases, edge cases, and one no-tool case
- A brief **evaluation** of what worked and what didn't

### Domain ideas
- **Study assistant** â€” grade conversion (French 0â€“20 â†” US A-F), GPA calculation, study-time estimator
- **Unit converter** â€” lengths, weights, temperatures, with context-aware explanations
- **Health tracker** â€” BMI, calorie estimation, heart-rate zones, VO2 max
- **Finance assistant** â€” interest calculation, savings projection, budget breakdown
- **Text analyser** â€” word count, reading time, keyword frequency, readability score
- **Your own idea** â€” describe it clearly in the design cell below

### 4.1 â€” âœï¸ Agent design

Before writing any code, describe your agent here.

### 4.1 — ✍️ Agent design

**Name:** Smart Personal Finance Assistant

**Domain:** Personal Finance & Wealth Management

**Who would use it and for what?**
> Individual users, students, or young professionals looking to structure their monthly budget, determine necessary emergency fund reserves, calculate investment growth, and evaluate the impact of inflation on purchasing power without performing manual arithmetic.

**List of tools (name + one-sentence description each):**
1. `budget_50_30_20` — Calculates allocation amounts for Needs (50%), Wants (30%), and Savings/Debt (20%) from net monthly income.
2. `calculate_emergency_fund` — Computes the recommended total emergency fund size based on essential monthly expenses and desired months of coverage.
3. `annual_to_monthly_rate` — Converts an annual interest rate percentage into a nominal monthly interest rate percentage.
4. `calculate_compound_interest` — Calculates future investment balance given initial principal, rate per period, and total compounding periods.
5. `calculate_inflation_impact` — Projects the future monetary cost required to retain purchasing power based on annual inflation.

**System prompt (draft):**
> You are a professional Personal Finance Assistant designed to help users plan budgets, calculate emergency funds, estimate investment growth, and understand financial concepts.
>
> OPERATIONAL RULES:
> 1. Always use available tools for calculations. Never compute compound interest, budget splits, or interest rates manually in your head.
> 2. For short-term investments specified in months with an annual interest rate, invoke `annual_to_monthly_rate` FIRST, then pass the resulting monthly rate to `calculate_compound_interest`.
> 3. If a tool fails with a ValueError (e.g., negative amounts or invalid parameters), explain the restriction clearly and politely to the user.
> 4. For general or conceptual questions that do not require calculation, answer directly without invoking any tools.
> 5. Present final answers clearly, formatted with currency units (€ or $) where applicable.

**6 test queries you plan to run:**
1. (normal case) "I earn 3500 EUR per month net. How should I split my money according to the 50/30/20 budget rule?"
2. (multi-step — budget + emergency fund) "My net income is 4000 EUR per month. Based on my 50% needs budget, how much should I keep in an emergency fund for 6 months?"
3. (edge case — error or boundary value) "Calculate my 50/30/20 budget for a net monthly income of -500 EUR."
4. (no tool needed — conceptual question) "What is the main difference between fixed-rate and variable-rate loans?"
5. (multi-step — auxiliary tool chain) "I want to invest 2000 EUR for 6 months at an annual interest rate of 12%. What will my final balance be?"
6. (inflation impact query) "If I buy an item today for 1000 EUR, how much will it cost in 5 years with an annual inflation rate of 3%?"

### 4.2 â€” âœï¸ Implement the tools

In [11]:
@tool
def budget_50_30_20(monthly_income: float) -> dict:
    """Divide a net monthly income according to the 50/30/20 budgeting rule.

    Args:
        monthly_income (float): Net monthly income amount. Must be greater than 0.

    Returns:
        dict: Allocation amounts for Needs (50%), Wants (30%), and Savings/Debt (20%).
    """
    if monthly_income <= 0:
        raise ValueError("Monthly income must be greater than zero.")

    return {
        "needs_50": round(monthly_income * 0.50, 2),
        "wants_30": round(monthly_income * 0.30, 2),
        "savings_20": round(monthly_income * 0.20, 2)
    }


@tool
def calculate_emergency_fund(monthly_expenses: float, months_coverage: int = 6) -> float:
    """Calculate the total recommended emergency fund size based on essential monthly expenses.

    Args:
        monthly_expenses (float): Essential monthly living expenses. Must be greater than 0.
        months_coverage (int): Number of months of expenses to cover. Must be at least 1. Default is 6.

    Returns:
        float: Total emergency fund amount needed.
    """
    if monthly_expenses <= 0 or months_coverage < 1:
        raise ValueError("Monthly expenses must be > 0 and months coverage must be >= 1.")

    return round(monthly_expenses * months_coverage, 2)


@tool
def annual_to_monthly_rate(annual_rate_percent: float) -> float:
    """Convert an annual interest rate percentage into a nominal monthly interest rate percentage.

    Args:
        annual_rate_percent (float): The annual interest rate in percent (e.g., 12.0 for 12%). Must be >= 0.

    Returns:
        float: The nominal monthly interest rate in percent (annual_rate / 12).
    """
    if annual_rate_percent < 0:
        raise ValueError("Annual interest rate cannot be negative.")

    return round(annual_rate_percent / 12.0, 4)


@tool
def calculate_compound_interest(principal: float, rate_percent_per_period: float, periods: int) -> float:
    """Calculate future value with compound interest given principal, interest rate per period, and total periods.

    Args:
        principal (float): Initial invested amount. Must be >= 0.
        rate_percent_per_period (float): Interest rate per period in percent (e.g., 1.0 for 1% per month or 10.0 for 10% per year). Must be >= 0.
        periods (int): Total number of compounding periods (e.g., months or years). Must be >= 1.

    Returns:
        float: Final accumulated balance (future value).
    """
    if principal < 0 or rate_percent_per_period < 0 or periods < 1:
        raise ValueError("Principal and rate must be >= 0, and periods must be at least 1.")

    rate_decimal = rate_percent_per_period / 100.0
    future_value = principal * ((1 + rate_decimal) ** periods)
    return round(future_value, 2)


@tool
def calculate_inflation_impact(current_amount: float, annual_inflation_rate_percent: float, years: int) -> float:
    """Calculate the future cost of a given monetary amount based on annual inflation rate.

    Args:
        current_amount (float): Present monetary value. Must be > 0.
        annual_inflation_rate_percent (float): Expected annual inflation rate in percent. Must be >= 0.
        years (int): Number of years in the future. Must be >= 1.

    Returns:
        float: Equivalent future cost needed to retain the same purchasing power.
    """
    if current_amount <= 0 or annual_inflation_rate_percent < 0 or years < 1:
        raise ValueError("Current amount must be > 0, inflation rate >= 0, and years >= 1.")

    rate_decimal = annual_inflation_rate_percent / 100.0
    future_cost = current_amount * ((1 + rate_decimal) ** years)
    return round(future_cost, 2)


# Combined tools list for Exercise 4
finance_tools = [
    budget_50_30_20,
    calculate_emergency_fund,
    annual_to_monthly_rate,
    calculate_compound_interest,
    calculate_inflation_impact
]

print(f"Finance Tools ({len(finance_tools)}): {[t.name for t in finance_tools]}")

Finance Tools (5): ['budget_50_30_20', 'calculate_emergency_fund', 'annual_to_monthly_rate', 'calculate_compound_interest', 'calculate_inflation_impact']


### 4.3 â€” âœï¸ Run the 5 test queries

In [12]:
MY_SYSTEM_PROMPT = """
You are a professional Personal Finance Assistant.
Your goal is to help users manage their personal finances, plan budgets, calculate investments, and estimate emergency funds.

OPERATIONAL RULES:
1. Always use available tools for calculations. Never calculate compound interest, budget splits, or rate conversions in your head.
2. If a query requires converting an annual interest rate to a monthly rate for a short-term investment (in months), execute `annual_to_monthly_rate` FIRST, then pass the output to `calculate_compound_interest`.
3. If a tool execution fails with a ValueError, explain the constraint politely to the user and suggest valid inputs.
4. If the user asks a general financial question that does NOT require math or tools (e.g., explaining a concept), answer directly without calling any tools.
5. Keep your final answers clear, professional, and formatted with currency units (e.g., € or $).
"""

def run_my_agent(query: str) -> str:
    """Convenience wrapper for your agent."""
    return react_agent(
        query=query,
        tools=finance_tools,
        system_prompt=MY_SYSTEM_PROMPT,
        max_steps=10,
        verbose=True,
    )

In [ ]:
# Query 1 â€” normal case
q1 = "I earn 3500 EUR per month net. How should I split my money according to the 50/30/20 budget rule?"
print(f"Query 1: {q1}")
r1 = run_my_agent(q1)
print(f"Answer: {r1}\n")

Query 1: I earn 3500 EUR per month net. How should I split my money according to the 50/30/20 budget rule?

--- STEP 1: REASON ---
  LLM content : (empty)
  Tool calls  : [{'name': 'budget_50_30_20', 'args': {'monthly_income': 3500}, 'id': 'e1c12103-475e-4e68-8d84-e80d7ddb2422', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  budget_50_30_20({'monthly_income': 3500}) â†’ {'needs_50': 1750.0, 'wants_30': 1050.0, 'savings_20': 700.0}

--- STEP 2: REASON ---
  LLM content : 'Based on the 50/30/20 budgeting rule, you should allocate your monthly income as'
  Tool calls  : []
  Msg count   : 5

--- LOOP EXITS: no more tool calls ---
Answer: Based on the 50/30/20 budgeting rule, you should allocate your monthly income as follows:
- **Needs (50%)**: €1,750
- **Wants (30%)**: €1,050
- **Savings/Debt Payments (20%)**: €700

These amounts will help you maintain a balanced budget.



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [14]:
# Query 2 â€” multi-step
q2 = "My net income is 4000 EUR per month. Based on my 50% needs budget, how much should I keep in an emergency fund for 6 months?"
print(f"Query 2: {q2}")
r2 = run_my_agent(q2)
print(f"Answer: {r2}\n")

Query 2: My net income is 4000 EUR per month. Based on my 50% needs budget, how much should I keep in an emergency fund for 6 months?

--- STEP 1: REASON ---
  LLM content : (empty)
  Tool calls  : [{'name': 'budget_50_30_20', 'args': {'monthly_income': 4000}, 'id': '1349dca1-3063-4409-a7d6-26de016da1da', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  budget_50_30_20({'monthly_income': 4000}) â†’ {'needs_50': 2000.0, 'wants_30': 1200.0, 'savings_20': 800.0}

--- STEP 2: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : 'Based on the 50/30/20 budgeting rule, for a net monthly income of 4000 EUR, your'
  Tool calls  : [{'name': 'calculate_emergency_fund', 'args': {'monthly_expenses': 2000, 'months_coverage': 6}, 'id': '09bc929b-6646-44dd-a504-40da123d7b5c', 'type': 'tool_call'}]
  Msg count   : 5

--- STEP 2: ACT + OBSERVE ---
  calculate_emergency_fund({'monthly_expenses': 2000, 'months_coverage': 6}) â†’ 12000.0

--- STEP 3: REASON ---
  LLM content : 'Based on a 6-month coverage period and essential monthly living expenses of 2000'
  Tool calls  : []
  Msg count   : 7

--- LOOP EXITS: no more tool calls ---
Answer: Based on a 6-month coverage period and essential monthly living expenses of 2000 EUR, your total recommended emergency fund size should be 12,000 EUR.



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [15]:
# Query 3 â€” edge case
q3 = "Calculate my 50/30/20 budget for a monthly income of -500 EUR."
print(f"Query 3: {q3}")
r3 = run_my_agent(q3)
print(f"Answer: {r3}\n")

Query 3: Calculate my 50/30/20 budget for a monthly income of -500 EUR.

--- STEP 1: REASON ---
  LLM content : 'An income of -500 EUR is not feasible as it suggests a loss or deficit. Please p'
  Tool calls  : []
  Msg count   : 3

--- LOOP EXITS: no more tool calls ---
Answer: An income of -500 EUR is not feasible as it suggests a loss or deficit. Please provide a net monthly income that is greater than 0 EUR.



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [16]:
# Query 4 â€” no tool needed
q4 = "What is the main difference between fixed-rate and variable-rate loans?"
print(f"Query 4: {q4}")
r4 = run_my_agent(q4)
print(f"Answer: {r4}\n")

Query 4: What is the main difference between fixed-rate and variable-rate loans?

--- STEP 1: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : 'The main difference between fixed-rate and variable-rate loans is that a fixed-r'
  Tool calls  : []
  Msg count   : 3

--- LOOP EXITS: no more tool calls ---
Answer: The main difference between fixed-rate and variable-rate loans is that a fixed-rate loan has a consistently scheduled interest rate over the loan term, which means the monthly payment amount remains the same throughout the loan period. In contrast, a variable-rate loan has an interest rate that can change over time, typically tied to an index like the Prime Rate. This can lead to both higher and lower payments depending on the movement of the index.



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [17]:
# Query 5 â€” your choice
q5 = "I want to invest 2000 EUR for 6 months. The annual interest rate is 12%. What will my final balance be?"
print(f"Query 5: {q5}")
r5 = run_my_agent(q5)
print(f"Answer: {r5}\n")

Query 5: I want to invest 2000 EUR for 6 months. The annual interest rate is 12%. What will my final balance be?

--- STEP 1: REASON ---


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  LLM content : (empty)
  Tool calls  : [{'name': 'annual_to_monthly_rate', 'args': {'annual_rate_percent': 12}, 'id': '290c9caf-e007-41d2-8444-d76c1e114fc1', 'type': 'tool_call'}, {'name': 'calculate_compound_interest', 'args': {'principal': 2000, 'rate_percent_per_period': 1, 'periods': 6}, 'id': 'd41d1e84-bf53-4000-93a9-0ab0431ca3d2', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  annual_to_monthly_rate({'annual_rate_percent': 12}) â†’ 1.0
  calculate_compound_interest({'principal': 2000, 'rate_percent_per_period': 1, 'periods': 6}) â†’ 2123.04

--- STEP 2: REASON ---
  LLM content : 'Given an initial investment of 2000 EUR with an annual interest rate of 12% over'
  Tool calls  : []
  Msg count   : 6

--- LOOP EXITS: no more tool calls ---
Answer: Given an initial investment of 2000 EUR with an annual interest rate of 12% over a period of 6 months, your final accumulated balance (future value) will be approximately 2123.04 EUR.



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [19]:
# Query 6 - Extra
q6 = "If I buy a item today for 1000 EUR, how much will it cost in 5 years with an annual inflation rate of 3%?"
print(f"Query 6: {q6}")
r6 = run_my_agent(q6)
print(f"Answer: {r6}\n")

Query 6: If I buy a item today for 1000 EUR, how much will it cost in 5 years with an annual inflation rate of 3%?

--- STEP 1: REASON ---
  LLM content : (empty)
  Tool calls  : [{'name': 'calculate_inflation_impact', 'args': {'current_amount': 1000, 'annual_inflation_rate_percent': 3, 'years': 5}, 'id': '87d8b8b9-a7bf-4ed9-9e99-ba4f87f2ac55', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  calculate_inflation_impact({'current_amount': 1000, 'annual_inflation_rate_percent': 3, 'years': 5}) â†’ 1159.27

--- STEP 2: REASON ---
  LLM content : 'In 5 years, with an annual inflation rate of 3%, the cost of a 1000 EUR item tod'
  Tool calls  : []
  Msg count   : 5

--- LOOP EXITS: no more tool calls ---
Answer: In 5 years, with an annual inflation rate of 3%, the cost of a 1000 EUR item today would be approximately 1159.27 EUR.



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


### 4.4 â€” âœï¸ Evaluation

Fill in the table and answer the questions.

### 4.4 — ✍️ Evaluation

**Evaluation table:**

| Query | Tools called | Correct? | Notes |
|-------|-------------|----------|-------|
| Q1 | `budget_50_30_20` | ✅ | Executed perfectly in 1 step and formatted the final answer cleanly in Markdown. |
| Q2 | `budget_50_30_20` → `calculate_emergency_fund` | ✅ | Perfect sequential chaining. Used the 'needs_50' value (€2,000) from Step 1 as input for the emergency fund calculation in Step 2. |
| Q3 | *None (reasoning-level rejection)* | ✅ | Successfully validated input (-500 EUR) directly via prompt/docstring constraints, refusing invalid tool invocation. |
| Q4 | *None* | ✅ | Respected Rule 4 of the System Prompt, directly answering a conceptual question without invoking tools. |
| Q5 | `annual_to_monthly_rate` + `calculate_compound_interest` | ✅ | Invoked both tools in parallel in Step 1. The model pre-calculated the monthly rate (1%) internally when passing arguments. |
| Q6 | `calculate_inflation_impact` | ✅ | Accurately computed the impact of 3% annual inflation over 5 years (€1,159.27) in a single step. |

---

### Questions

1. What is the most important thing you learned about writing effective tool docstrings?
   > **Docstrings serve as the prompt specification and JSON Schema contract for the LLM.**
   > - When utilizing `bind_tools()`, the function signature and docstring are converted into a structured schema that guides the model on *when* and *how* to invoke the tool.
   > - **Evidence from logs:** In **Query 3** (`-500 EUR`), the model did not even need to execute Python code to discover that a negative income was invalid. The constraint specified in the docstring (*"Must be greater than 0"*) was parsed during the LLM's reasoning step, prompting it to reject the input and inform the user immediately before running any code.

2. Where did your agent fail or surprise you? What would you change?
   > **Surprises & Notable Behaviors:**
   > - **Positive Surprise (Q3):** The agent demonstrated proactive input validation by declining tool execution for logically impossible inputs (-500 EUR income), saving unnecessary tool invocations and execution overhead.
   > - **Speculative Parallel Invocation (Q5):** In Query 5, the intention was to force a step-by-step execution (Step 1: convert annual rate to monthly; Step 2: calculate compound interest). However, `qwen2.5:3b` triggered **both tools simultaneously in Step 1**. Remarkably, it performed the mental calculation $12\% / 12 = 1\%$ and supplied `rate_percent_per_period: 1` directly to the compound interest tool in the same turn.
   > 
   > **What I would change:**
   > - To prevent the LLM from attempting to guess or pre-compute arguments for dependent tools in parallel, I would set `parallel_tool_calls=False` in the binding layer (if available) or enforce a strict system prompt constraint: *"NEVER estimate or pre-calculate arguments for dependent tools in parallel; execute tools sequentially one step at a time."*

3. How does this ReAct agent relate to the four agentic patterns from CM1?
   > The implemented ReAct agent directly incorporates the 4 fundamental agentic design patterns:
   > 1. **Tool Use:** The core mechanism of ReAct. The LLM expands beyond text generation by executing deterministic Python functions for accurate numerical operations.
   > 2. **Planning:** Demonstrated in the *Reasoning + Acting* loop. In **Query 2**, the agent decomposed the task into two logical steps: first determining essential living expenses (50%), then using that output to calculate the 6-month emergency fund.
   > 3. **Reflection / Self-Correction:** The agent continuously evaluates tool observations against context history to decide whether the user's objective has been met or if additional tool calls are required.
   > 4. **Multi-Agent / Delegation:** Although structured as a single orchestrator loop, the agent delegates specialized execution sub-tasks to deterministic Python modules acting as tool executors.

4. In TP2 your agent will retrieve information from documents (RAG) instead of calling Python functions. What do you think will change in the loop? What stays the same?
   > **What stays the SAME:**
   > - The underlying **ReAct** execution loop (`Prompt → Thought/Reason → Act → Observe → Output`).
   > - Context and message history management (`SystemMessage`, `HumanMessage`, `ToolMessage`/`ContextMessage`).
   > - The requirement for explicit `SystemPrompt` guidelines to prevent hallucination and enforce clear stopping conditions.
   >
   > **What CHANGES:**
   > - **Tool Nature:** Instead of calling deterministic mathematical Python functions, tools will act as *Retrievers* executing semantic vector searches over document stores.
   > - **Observation Nature:** Tool outputs will transition from structured scalar or dictionary outputs to unstructured text chunks extracted from documents.
   > - **Synthesis & Hallucination Challenge:** In RAG workflows, the primary challenge shifts to accurately extracting, synthesizing, and reasoning over noisy textual context without introducing ungrounded claims, necessitating precise source attribution.

---
## What comes next

| Session | Date | You will build |
|---------|------|----------------|
| TP2 | Nov 10â€“13 | RAG pipeline â€” retrieval as a tool, ChromaDB, RAGAS evaluation |
| TP3+TP4 | Nov 18 | Multi-agent system with CrewAI + intro to LangGraph |
| EDT1â€“4 | Nov 23 â†’ Dec 8 | Your group project â€” agent on a real business problem |
| Soutenance | Dec 9 | Demo + Q&A |
| QCM | Dec 10 | 1h â€” key concepts from the module |